In [ ]:
!pip install beautifulsoup4 requests

In [ ]:
import re
import requests
from pathlib import Path
from bs4 import BeautifulSoup

# グローバル変数の定義
BASE_DOMAIN = "https://www.ipa.go.jp"
START_URL = "https://www.ipa.go.jp/shiken/mondai-kaiotu/index.html"
SC_HEADINGS = ["情報セキュリティスペシャリスト試験（SC）", "情報処理安全確保支援士試験（SC）"]

url = START_URL
res = requests.get(url)
res.encoding = res.apparent_encoding
soup = BeautifulSoup(res.text, "html.parser")

# 「問題冊子・配点割合・解答例・採点講評」を含むリンクをすべて抽出
links = {}
for a in soup.find_all("a"):
    if a.text and "問題冊子・配点割合・解答例・採点講評" in a.text:
        href = a.get("href")
        # 絶対URL化
        if href and not href.startswith("http"):
            href = BASE_DOMAIN + href
        links[a.text.strip()] = href

# 辞書型として格納
for title, link in links.items():
    print(f"{title}: {link}")


In [ ]:
pdf_links = {}

def extract_year(year_title):
    m = re.search(r'(\d{4})年度', year_title)
    return m.group(1) if m else ''

def extract_term(text):
    if '春期' in text: return '春期'
    if '秋期' in text: return '秋期'
    if '通年' in text: return '通年'
    return ''

def extract_info_from_filename(filename):
    # 例: 2025r07h_koudo_am1_qs.pdf
    # 区分辞書
    exam_map = {
        'sc': '情報処理安全確保支援士試験（SC）',
        'koudo': '情報処理安全確保支援士試験（SC）',
        'scsp': '情報セキュリティスペシャリスト試験（SC）',
    }
    subpart_map = {
        'am1': '午前I',
        'am2': '午前II',
        'pm1': '午後I',
        'pm2': '午後II',
        'am': '午前',
        'pm': '午後',
    }
    exam = ''
    subpart = ''
    m_exam = re.search(r'_(sc|koudo|scsp)_', filename)
    if m_exam:
        exam = exam_map.get(m_exam.group(1), m_exam.group(1))
    m_sub = re.search(r'_(am1|am2|pm1|pm2|am|pm)_', filename)
    if m_sub:
        subpart = subpart_map.get(m_sub.group(1), m_sub.group(1))
    return exam, subpart

def is_unwanted_pdf(text, filename):
    return any(x in text for x in ['要綱', '配点割合']) or 'youkou' in filename





for year_title, url in links.items():
    res = requests.get(url)
    res.encoding = res.apparent_encoding
    soup = BeautifulSoup(res.text, "html.parser")
    year = extract_year(year_title)

    current_term = '通年'
    h3_tags = soup.find_all(['h3'])
    for h3 in h3_tags:
        term = extract_term(h3.get_text())
        if term:
            current_term = term
        sib = h3
        while True:
            sib = sib.find_next_sibling()
            if sib is None or sib.name == 'h3':
                break
            if sib.name in ['h4','h5','h6','strong','b'] and any(h in sib.get_text() for h in SC_HEADINGS):
                for a in sib.find_all_next('a'):
                    stop_tag = a.find_previous(['h3','h4','h5','h6','strong','b'])
                    if stop_tag and stop_tag != sib and stop_tag.name in ['h3','h4','h5','h6','strong','b']:
                        break
                    if a.get('href','').endswith('.pdf'):
                        text = a.get_text(strip=True)
                        href = a.get('href')
                        if href and not href.startswith('http'):
                            href = BASE_DOMAIN + href
                        filename = href.split('/')[-1]
                        if is_unwanted_pdf(text, filename):
                            continue
                        booklet = re.split(r'\(|（', text)[0].strip()
                        exam, subpart = extract_info_from_filename(filename)
                        key_parts = [year, current_term, exam, subpart, booklet]
                        key = '_'.join([k for k in key_parts if k]) + '.pdf'
                        pdf_links[key] = href
                        

for i, (key, link) in enumerate(pdf_links.items()):  # 最初の10件だけ表示
    print(f'{key}: {link}')
    if i >= 9:
        break


In [ ]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

for key, url in pdf_links.items():
    save_path = output_dir / key
    try:
        res = requests.get(url)
        res.raise_for_status()
        with open(save_path, "wb") as f:
            f.write(res.content)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Failed to download {url}: {e}")
